# 策略概述

**DTW** 以「動態時間校正」距離衡量兩檔股票的走勢相似度——相較於逐日對齊的 SSD，它能**容忍兩股走勢有輕微的時間錯位**（例如一檔領先、另一檔幾天後才跟上）。

在同一產業內，先確認兩股價差通過共整合檢定（有均值回歸性質），再依 DTW 距離挑出走勢最相似的配對。它是本研究比較「距離衡量方式」時的重要基準之一。


# 策略架構

與傳統距離法相同的四層管線，差別在**排序層改用能容忍時間錯位的 DTW 距離**。

```{mermaid}
flowchart LR
  P["形成期日價格<br/>標準化"] --> G["分組<br/>GICS 產業"]
  G --> F["篩選<br/>共整合 + 半衰期 + Hurst"]
  F --> R["排序<br/>DTW 時間校正距離"]
  R --> T["Top N 配對<br/>→ 交易期"]
```

| 層 | 本策略採用 | 用途 |
| :--- | :--- | :--- |
| 分組 | GICS 產業分類 | 同產業候選 |
| 篩選 | 共整合 + 半衰期 + Hurst | 確認價差均值回歸 |
| 排序 | Sakoe-Chiba 限制窗 DTW 距離 | 容忍時間錯位的走勢相似度 |
| 交易 | Z-Score（標準化空間重建價差） | 偏離進場、回歸出場 |


# 參考文獻與引用對應


## 文獻 1：許鈞翔 (2025)

> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。碩士論文。

**參考部分**：

- 整體形成期流程設計：**「共整合檢定先行篩選 → 對通過者計算距離 → 依距離排序選取」**的三段式架構
- 以 **DTW（動態時間校正）距離**取代純歐氏距離作為走勢相似度量度
- 形成期／交易期視窗與滾動回測設計

**為何參考**：

- 本模組即為此論文方法的**對齊實作**（模組 docstring 明載「許鈞翔 (2025) 論文對齊版」），流程順序、篩選條件與排序方式皆按論文重現
- 「先共整合、後距離」的順序讓昂貴的 DTW 計算只發生在已確認具均衡關係的配對上



## 文獻 2：Sakoe & Chiba (1978)

> Sakoe, H., & Chiba, S. (1978). Dynamic programming algorithm optimization for spoken word recognition. *IEEE Transactions on Acoustics, Speech, and Signal Processing, 26*(1), 43–49.

**參考部分**：

- DTW 動態規劃演算法本體（insertion／deletion／match 三向遞迴）
- **Sakoe-Chiba 帶（band）約束**：限制對齊路徑偏離對角線的最大距離

**為何參考**：

- 本策略的 DTW 實作（Sakoe-Chiba 限制窗 DTW 距離）直接採用其帶約束形式：把時間扭曲限制在 $W = 15$ 天內，防止兩段相隔數月的走勢被不合理地對齊，同時把計算複雜度從 $O(N^2)$ 降至 $O(N \cdot W)$



## 文獻 3：Engle & Granger (1987)

> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction: Representation, estimation, and testing. *Econometrica, 55*(2), 251–276.

**參考部分**：

- 兩步驟共整合檢定程序：OLS 估計均衡關係 → 對殘差做 ADF 單根檢定

**為何參考**：

- 本策略階段 3 的「雙向 OLS + ADF」即此程序的實作；因 Engle-Granger 檢定結果**依賴回歸方向**（以 A 回歸 B 與以 B 回歸 A 的殘差不同），本策略對兩個方向各檢定一次、取 p 值較小者，確保不因方向選擇錯過共整合配對



## 文獻 4：Krauss, Do & Huck (2016)

> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies: Distance, cointegration and copula methods. *European Journal of Operational Research*.

**參考部分**：

- **OU 半衰期**與 **Hurst 指數**作為均值回歸品質的量化指標

**為何參考**：

- 統計過濾第二道（半衰期 $1$–$42$ 日）與第三道（$H < 0.5$）的設計依據，確保通過共整合檢定的配對，其回歸速度落在交易期內可實現的時間尺度



# 各階段行為

策略在每個滾動形成窗（252 交易日，每 21 日滾動）內依序執行以下七個階段。


## 階段 1：資料範圍界定與產業分組

**輸入**：形成窗內全部 S&P 500 歷史成分股的日收盤價矩陣。

**行為**：

1. 依 GICS 產業分類將股票分組，配對搜尋只在同產業內進行
2. 產業標記為 `Unknown` 的股票整組跳過
3. 股票數不足 `min_tickers_for_pairing`（= 2）的產業跳過

**設計理由**：同產業股票共享產業層級風險因子，均衡關係具經濟基礎。


## 階段 2：對數價格 Z-Score 標準化

對形成窗內每支股票 $i$：

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_{\ln P_i}}{\sigma_{\ln P_i} + \varepsilon}, \qquad \varepsilon = 10^{-12}$$

取對數前以 $\max(P, 10^{-8})$ 下限保護。標準化後每股序列均值 0、標準差 1，後續的 OLS、SSD 與 DTW 皆在此標準化空間中計算。

標準化統計量（`Log_Mean_A/B`、`Log_Std_A/B`）隨配對輸出，供交易期重建同一座標。


## 階段 3：雙向 OLS 回歸與方向決定

Engle-Granger 檢定結果依賴回歸方向，因此對每對股票 $(u, v)$ **兩個方向各做一次** OLS（標準化空間，含截距）：

$$\text{方向 1：} P'_{u,t} = \alpha_1 + \beta_1 P'_{v,t} + \epsilon^{(1)}_t \qquad
\text{方向 2：} P'_{v,t} = \alpha_2 + \beta_2 P'_{u,t} + \epsilon^{(2)}_t$$

分別對殘差 $\epsilon^{(1)}, \epsilon^{(2)}$ 做 ADF 檢定（`max_lags=1`），**取 p 值較小的方向**：

$$(\text{Ticker\_A},\ \text{Ticker\_B}) = \arg\min_{\text{方向}} p_{ADF}$$

被解釋變數記為 Ticker_A、解釋變數記為 Ticker_B，並保留該方向的 $\alpha$（`OLS_Alpha`）、$\beta$（`Hedge_Ratio`）與殘差序列。


## 階段 4：三道統計過濾

對選定方向的殘差 spread 依序檢定，任一道未通過即淘汰。

### 第一道：ADF 共整合（依據：Engle & Granger 1987）

$$\text{要求 } p_{ADF} < 0.01$$

顯著水準取 0.01：因後續距離計算成本高（DTW 為逐對動態規劃），以較嚴格的門檻先縮小候選集。

### 第二道：OU 半衰期（依據：Krauss et al. 2016）

對 $\Delta \epsilon_t = c + \lambda\, \epsilon_{t-1} + u_t$ 做最小平方估計：

$$\lambda < 0, \qquad HL = \frac{-\ln 2}{\lambda}, \qquad 1 \le HL \le \frac{126}{3} = 42 \text{ 日}$$

### 第三道：Hurst 指數（依據：Krauss et al. 2016）

對 spread 做 R/S 分析（`already_stationary=True`）：

$$H < 0.50$$


## 階段 5：SSD 與 DTW 距離計算

**只對通過全部統計檢定的配對**計算距離（節省計算）。

**SSD**（同步逐日比較）：

$$\text{SSD}_{A,B} = \sum_{t=1}^{F} \left(P'_{A,t} - P'_{B,t}\right)^2$$

**Sakoe-Chiba DTW**（依據：Sakoe & Chiba 1978；容許時間錯位的比較）：

$$D(i,j) = (P'_{A,i} - P'_{B,j})^2 + \min\big\{ D(i-1,j),\ D(i,j-1),\ D(i-1,j-1) \big\}$$

$$\text{DTW}_{A,B} = D(F, F), \qquad \text{約束 } |i - j| \le W = 15$$

- DTW 容許兩股走勢存在**領先／落後（lead-lag）**關係：一股先漲、另一股數日後跟上，同步比較（SSD）會放大此差異，DTW 則能沿最佳對齊路徑吸收
- 帶寬 $W = 15$ 天：超過三週的走勢錯位不再視為同一均衡關係的表現


## 階段 6：排序與配對選取

全部通過檢定的配對依 **DTW 距離升序**排列，取前 `top_n` 組（`method="dtw"`）。

每組輸出欄位：

| 欄位 | 內容 | 交易期用途 |
| :--- | :--- | :--- |
| `Ticker_A` / `Ticker_B` | ADF 最佳方向的配對代碼 | 建倉標的 |
| `Sector` | GICS 產業 | 產業分散控管 |
| `SSD` / `DTW_Dist` | 兩種距離值 | 排序依據（記錄用） |
| `Hedge_Ratio` | 最佳方向 OLS 斜率 $\beta$ | spread 重建與部位配重 |
| `OLS_Alpha` | 最佳方向 OLS 截距 $\alpha$ | 交易期**不使用**（見階段 7） |
| `Spread_Mean` / `Spread_Std` | 形成期殘差均值／標準差 | Z-Score 中心與分母 |
| `Log_Mean_A/B`、`Log_Std_A/B` | 形成期對數價格統計量 | 交易期標準化座標 |


## 階段 7：交易期的參數使用方式（座標修正）

本策略的 OLS 在**標準化空間**擬合，因此交易期必須在同一空間重建 spread。config 以 `ignore_ols_alpha=True` 指示交易端**忽略 `OLS_Alpha`**、直接使用標準化座標：

$$P'_{i,t} = \frac{\ln P_{i,t} - \texttt{Log\_Mean}_i}{\texttt{Log\_Std}_i}, \qquad
\text{Spread}_t = P'_{A,t} - \texttt{Hedge\_Ratio} \cdot P'_{B,t}$$

$$Z_t = \frac{\text{Spread}_t - \texttt{Spread\_Mean}}{\texttt{Spread\_Std}}$$

**座標一致性說明**：若交易期改在原始 log-price 空間使用 $\ln P_A - \alpha - \beta \ln P_B$ 重建 spread，會與形成期的標準化空間座標錯位，產生恆定的 Z-Score 偏移（歷史版本曾有此問題，`ignore_ols_alpha` 即為修正機制，策略名稱的「Fixed」由此而來）。

形成期統計量整個交易期凍結不變（無前視）。交易決策細節見 `trading/zscore_trading.ipynb`。


# 參數總表

| 參數 | 值 | 對應階段 | 說明 |
| :--- | :---: | :--- | :--- |
| 形成窗 / 滾動步長 | 252 / 21 交易日 | 全流程輸入 | 約一年 / 一個月 |
| 每期配對數 | 網格 [1, 3, 5, 10, 20] | 排序（選取） | 依 DTW 距離升序取前幾組 |
| 距離衡量 | DTW | 排序 | 動態時間校正距離 |
| DTW 帶寬 | 15 天 | 排序 | Sakoe-Chiba 限制窗，避免過度扭曲對齊 |
| 共整合顯著水準 | 0.01 | 篩選 | 價差通過共整合檢定門檻 |
| 半衰期範圍 | $[1,\ 42]$ 日 | 篩選 | 回歸速度合理區間 |
| Hurst 上限 | 0.50 | 篩選 | 均值回歸判準 |
| 價差空間 | 標準化空間 | 交易 | 與形成期一致，避免座標偏移 |
